In [ ]:
import os
import math
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from classifier import MLP
from datasets.MNISTPerClass import MNISTPerClass
from datasets.MNIST import MNIST
from Autoencoder_real import KoopmanAutoencoder
from Autoencoder_functions import koopman_loss, collect_latent_states
from torch.nn.utils import parameters_to_vector
from scipy.linalg import eig, inv
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

import optuna

In [ ]:

def compute_gradient_norm(model, norm_type=2):
    with torch.no_grad():
        total_norm = 0.0
        for param in model.parameters():
            if param.grad is not None:
                param_norm = param.grad.norm(norm_type)
                total_norm += param_norm.item() ** norm_type
        total_norm = total_norm ** (1.0 / norm_type)
    return total_norm

def classifier_sub(x, p_vec, classifier_shapes):
    '''
    The forward propgation in this function must be the same as in the classifier
    '''
    idx, p_recon = 0, []
    for layer in classifier_shapes:
        layer_params = []
        for shape in layer:
            offset = np.prod(shape)
            layer_params.append(p_vec[idx:idx+offset].reshape(shape))
            idx += offset
        p_recon.append(layer_params)
    
    w0, b0 = p_recon[0]
    w1, b1 = p_recon[1]
    x = F.linear(x, w0, b0)
    x = F.relu(x)
    x = F.linear(x, w1, b1)
    return x

def compute_l_classifier(model, images, labels, criterion_classifier):
    # Move tensors to device
    images = images.reshape(-1, 28*28).to(device)
    labels = labels.to(device)

    # Forward pass
    outputs = model(images)
    loss = criterion_classifier(outputs, labels)
    return loss

def compute_l_kae(kae, params_snapshots, p, c1, c2, c3):
    x = torch.stack(params_snapshots, dim=0).to(device)
    latents, latents_next = collect_latent_states(kae, x)
    kae.compute_koopman_operator(latents, latents_next)
    x_hat, z, z_pred = kae(x)
    recon_loss, state_pred_loss, koopman_pred_loss = koopman_loss(x, x_hat, z_pred, p, kae)
    loss_kae = c1*recon_loss + c2*state_pred_loss + c3*koopman_pred_loss # + c4*k_norm_loss      
    return loss_kae, z


def compute_theta_sub_all(kae, z):
    ko = kae.K
    eigvals, eigvec_left = torch.linalg.eig(ko)
    eigvec_left = eigvec_left.real.detach()
    # for e in range(hidden_k):
    #     writer.add_scalar(f'Eigval/{e}', torch.abs(eigvals[e]), n_batch * epoch + inner)
    # B = np.pad(np.eye(n_params), ((0, 0), (0, N_O - n_params)), mode='constant')
    eigvec_left_inv = torch.linalg.pinv(eigvec_left)
    v = (kae.decoder(eigvec_left_inv)).T
    phi = eigvec_left @ z[-1, :]
    param_sub_all = v @ torch.diag(phi)
    return param_sub_all, eigvals

def compute_l_sub(param_sub, target_classes, images, labels, classifier_shapes, criterion_classifier):
    # classifier_sub = MLP(image_size, hidden_c, num_classes).to(device)
    # nn.utils.vector_to_parameters(param_sub, classifier_sub.parameters())
    images = images.reshape(-1, 28*28).to(device)
    labels = labels.to(device)
    mask = torch.isin(labels, target_classes.clone().detach())
    images = images[mask]
    labels = labels[mask]
    outputs = classifier_sub(images, param_sub, classifier_shapes)
    loss = criterion_classifier(outputs, labels)
    return loss

def test_classifier(model, test_loader):
    model.eval()  # evaluation mode
    with torch.no_grad():
        correct = 0
        total = 0
        for images, labels in test_loader:
            images = images.reshape(-1, 28*28).to(device)
            labels = labels.to(device)
            outputs = model(images)
            predicted = torch.argmax(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total
        print(f'Test Accuracy: {accuracy:.2f}%')

def get_target_classes(param_sub, candidates, images, labels, num_class_per_mode, classifier_shapes):
    with torch.autograd.no_grad():
        images = images.reshape(-1, 28*28).to(device)
        labels = labels.to(device)
        outputs = classifier_sub(images, param_sub, classifier_shapes)
        _, pred = torch.max(outputs, 1)
        acc = []
        for i in candidates:
            if i != -1:
                mask = labels == i
                acc.append((pred[mask] == labels[mask]).sum()/mask.sum())
            else:
                acc.append(-1)
        _, top_idx = torch.topk(torch.tensor(acc), num_class_per_mode)
        return candidates[top_idx]
        

In [ ]:
seed = 5
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ['PYTHONHASHSEED'] = str(seed)

save = False
c1 = 1
c2 = 1
c3 = 1

hidden_c = 16
image_size = 784  # 28x28 images flattened
num_classes = 10
batch_size = 128
num_epochs = 10 #####################################################################################################################################################
n_trials = 10 #####################################################################################################################################################
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load datasets
mnist_per_class = MNISTPerClass(batch_size=batch_size)
mnist = MNIST(batch_size=batch_size)


def objective(trial):
    lr_kae_pw = trial.suggest_int("lr_kae_pw", -6, -2)
    lr_kae = 10 ** lr_kae_pw
    lr_classifier_pw = trial.suggest_int("lr_classifier_pw", -6, -2)  
    lr_classifier = 10 ** lr_classifier_pw
    hidden_k = trial.suggest_int('hidden_k', 6, 128)
    num_mode_dom = trial.suggest_int('num_mode_dom', 3, 6)
    T = trial.suggest_int('T', 5, 5) #######################################################################################################################################
    p = trial.suggest_int('p', 5, 20)
    kae_coef = trial.suggest_int("kae_coef", 1, 1000)
    sub_coef = trial.suggest_int("sub_coef", 1, 1000)
    max_param_stack = trial.suggest_int('max_param_stack', 5, 10) ##########################################################################################################

    num_class_per_mode = int(math.ceil(num_classes/num_mode_dom))

    # Build the classifier
    classifier = MLP(image_size, hidden_c, num_classes).to(device)
    classifier.train()
    criterion_classifier = nn.CrossEntropyLoss()
    optimizer_classifier = optim.Adam(classifier.parameters(), lr=lr_classifier)
    classifier_shapes = [
        [(hidden_c, image_size), (hidden_c)],
        [(num_classes, hidden_c), (num_classes)]
    ]    
    
    # Build the KAE
    param_vec = parameters_to_vector(classifier.parameters()) 
    state_dim = param_vec.shape[0]

    kae = KoopmanAutoencoder(state_dim=state_dim, hidden_dim=hidden_k).to(device)
    kae.train()
    criterion_kae = koopman_loss
    mse = torch.nn.MSELoss()
    optimizer_kae = optim.Adam(kae.parameters(), lr=lr_kae)

    # Get T number of snapshots first
    train_loader_classifier = mnist.train_loader
    test_loader = mnist.test_loader
    params_snapshots = []
    # print('new MNIST training')
    for epoch in tqdm(range(T)):
        running_loss = 0.0
        params_snapshots.append(parameters_to_vector(classifier.parameters()))
        for images, labels in train_loader_classifier:
            # train_loader_per_class = mnist_per_class.sub_trainloaders[epoch%10]
            loss_classifier = compute_l_classifier(classifier, images, labels, criterion_classifier)
            optimizer_classifier.zero_grad()
            loss_classifier.backward()
            optimizer_classifier.step()
            running_loss += loss_classifier.item()
        # print(f'Epoch [{epoch+1}/{T}], Loss: {running_loss/len(train_loader_classifier):.4f}')
    test_classifier(classifier, test_loader)


    n_params = len(params_snapshots[0])
    # print(n_params)
    classifier.train()
    kae.train()
    n_batch = len(train_loader_classifier)
    for epoch in tqdm(range(num_epochs)):
        running_loss = 0.0
        for inner, (images, labels) in enumerate(train_loader_classifier):
            loss_sub = 0.0
            loss_classifier = compute_l_classifier(classifier, images, labels, criterion_classifier)
            loss_kae, z = compute_l_kae(kae, params_snapshots, p, c1, c2, c3) # Koopman operator is updated here.
            N_O = z.shape[-1]
            param_sub_all, eigvals = compute_theta_sub_all(kae, z)

            order = torch.argsort(eigvals.abs())
            candidates = torch.linspace(0, 9, 10, dtype=int, device=device)
            for _ in range(num_mode_dom):
                best_mode = torch.argmax(order)
                order[best_mode] = -1
                target_classes = get_target_classes(param_sub_all[:, best_mode], candidates, images, labels, num_class_per_mode, classifier_shapes)
                if -1 in candidates[target_classes]:
                    rest = ~(candidates == -1)
                    target_classes = candidates[rest]
                candidates[target_classes] = -1

                loss_sub = loss_sub + compute_l_sub(param_sub_all[:, best_mode], target_classes, images, labels, classifier_shapes, criterion_classifier)


            loss = kae_coef * loss_kae + sub_coef * loss_sub  # loss_classifier + loss_kae +

            optimizer_kae.zero_grad()
            loss.backward()
            optimizer_kae.step()
            ##############################
            optimizer_classifier.zero_grad()
            loss_classifier.backward() 
            optimizer_classifier.step()
            ##############################
            params_snapshots.append(parameters_to_vector(classifier.parameters()))
            params_snapshots = params_snapshots[-(max_param_stack-1):]
            running_loss += loss.item()
        # print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader_classifier):.4f}')
    return running_loss/len(train_loader_classifier)

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=n_trials)

# # Test the original classifier again
# test_classifier(classifier, test_loader)


In [ ]:
import optuna.visualization as vis

# Plot optimization history
vis.plot_optimization_history(study).show()

# Plot parameter importance
vis.plot_param_importances(study).show()

# Parallel coordinate plot
vis.plot_parallel_coordinate(study).show()

# Slice plot: value vs parameter
vis.plot_slice(study).show()

# Contour plot (2D param relationship)
vis.plot_contour(study).show()
